
# TMDB Movie Data Pipeline & EDA
**Name:** J. Harshavardhan  
**Assignment:** Data Pipeline using TMDB API + Exploratory Data Analysis  

---

## About this Notebook
In this notebook, I built a small data pipeline that fetches movie data from the TMDB (The Movie Database) API, stores it in a SQLite database, and then performs Exploratory Data Analysis (EDA) on the collected data using pandas.

The main goals are:
- Fetch at least 20 movies using the TMDB Discover Movies endpoint
- Store the data in a SQLite database table called `movies`
- Load it back and analyze it using pandas

## Step 1 — Importing Libraries & Setup

First I imported the necessary libraries. I used `requests` to call the TMDB API, `pandas` to work with the data, and `sqlite3` (which is built into Python, no installation needed) to store the data locally.

The API key is stored securely in Google Colab's Secrets feature so it doesn't get exposed in the code.

In [19]:
import requests
import pandas as pd
import sqlite3
from google.colab import userdata

API_KEY = userdata.get("TMDB_API_KEY")

BASE_URL = "https://api.themoviedb.org/3"

## Step 2 — Fetching Movie Data from TMDB API

Here I used the TMDB **Discover Movies** endpoint to pull trending/popular movies. I fetched **2 pages × 20 results = 40 movies** in total which is more than the required minimum of 20.

For each movie I extracted 8 fields:
- `movie_id` — unique TMDB ID
- `title` — name of the movie
- `release_date` — when it was released
- `popularity` — TMDB popularity score
- `vote_average` — average rating out of 10
- `vote_count` — total number of votes
- `overview` — short description
- `genres` — genre names (converted from IDs using a mapping dictionary)

I also added error handling — if any page fails, the script prints the error and moves on instead of crashing.

In [20]:
# TMDB genre ID mapping (to convert IDs to names later)
GENRE_MAP = {
    28: "Action", 12: "Adventure", 16: "Animation",
    35: "Comedy", 80: "Crime", 99: "Documentary",
    18: "Drama", 10751: "Family", 14: "Fantasy",
    36: "History", 27: "Horror", 10402: "Music",
    9648: "Mystery", 10749: "Romance", 878: "Science Fiction",
    10770: "TV Movie", 53: "Thriller", 10752: "War", 37: "Western"
}

all_movies = []  # store all fetched movies here

# Fetch 2 pages x 20 results = at least 40 movies
for page in range(1, 3):
    url = f"{BASE_URL}/discover/movie"

    params = {
        "api_key": API_KEY,
        "language": "en-US",
        "sort_by": "popularity.desc",
        "page": page
    }

    response = requests.get(url, params=params)

    # Check if request was successful
    if response.status_code != 200:
        print(f"Error on page {page}: {response.status_code}")
        continue

    data = response.json()
    movies = data.get("results", [])

    for movie in movies:
        # Convert genre IDs to readable names
        genre_names = [GENRE_MAP.get(gid, "Unknown") for gid in movie.get("genre_ids", [])]

        all_movies.append({
            "movie_id":     movie.get("id"),
            "title":        movie.get("title"),
            "release_date": movie.get("release_date"),
            "popularity":   movie.get("popularity"),
            "vote_average": movie.get("vote_average"),
            "vote_count":   movie.get("vote_count"),
            "overview":     movie.get("overview"),
            "genres":       ", ".join(genre_names)  # store as comma-separated string
        })

print(f"Fetched {len(all_movies)} movies total")

Fetched 40 movies total


## Step 3 — Creating the DataFrame

I converted the list of movie dictionaries into a pandas DataFrame. This makes it easy to view, clean, and analyze the data. The shape `(40, 8)` confirms we have 40 movies and 8 columns.

In [21]:
df = pd.DataFrame(all_movies)
print(f"Shape: {df.shape}")
df.head()

Shape: (40, 8)


,movie_id,title,release_date,popularity,vote_average,vote_count,overview,genres
0,1523145,Your Heart Will Be Broken,2026-03-26,1165.1266,6.648,27,High school student Polina is saved from bully...,"Romance, Drama"
1,83533,Avatar: Fire and Ash,2025-12-17,503.6662,7.327,2109,In the wake of the devastating war against the...,"Science Fiction, Adventure, Fantasy"
2,1115544,Mike & Nick & Nick & Alice,2026-03-14,352.9397,6.732,114,Two gangsters and the woman they love try to s...,"Comedy, Science Fiction, Crime"
3,1084187,Pretty Lethal,2026-03-13,311.0726,6.765,162,A troupe of ballerinas find themselves fightin...,"Action, Thriller, Music"
4,1290821,Shelter,2026-01-28,308.3847,6.751,445,A man living in self-imposed exile on a remote...,"Action, Crime, Thriller"


## Step 4 — Saving to SQLite Database

I saved the DataFrame into a local SQLite database file called `tmdb.db` under a table named `movies`. SQLite is a lightweight database that doesn't need any server setup — perfect for small projects like this.

In [22]:
# Load back from SQLite
conn = sqlite3.connect("tmdb.db")
df_loaded = pd.read_sql("SELECT * FROM movies", con=conn)
print(f"Loaded {len(df_loaded)} rows from SQLite")
conn.close()

Loaded 40 rows from SQLite


## Step 5 — Loading Data Back from SQLite

To verify the data was stored correctly, I loaded the `movies` table back from the SQLite database into a new DataFrame called `df_loaded`.

In [23]:
conn = sqlite3.connect("tmdb.db")
df_loaded = pd.read_sql("SELECT * FROM movies", con=conn)
print(f"Loaded {len(df_loaded)} rows from SQLite")
conn.close()

Loaded 40 rows from SQLite


---

# Task 2 — Exploratory Data Analysis (EDA)

Now that the data is stored and loaded, I performed basic EDA to understand what the dataset looks like — its structure, statistics, and any issues like missing values.

## EDA 1 — First 5 Rows & Summary Statistics

`df.head()` shows the first 5 rows to get a quick look at the data. `describe()` gives a statistical summary of all numeric columns like mean, min, max, and standard deviation.

In [24]:
# First 5 rows
print("First 5 rows:")
df_loaded.head()

First 5 rows:


,movie_id,title,release_date,popularity,vote_average,vote_count,overview,genres
0,1523145,Your Heart Will Be Broken,2026-03-26,1165.1266,6.648,27,High school student Polina is saved from bully...,"Romance, Drama"
1,83533,Avatar: Fire and Ash,2025-12-17,503.6662,7.327,2109,In the wake of the devastating war against the...,"Science Fiction, Adventure, Fantasy"
2,1115544,Mike & Nick & Nick & Alice,2026-03-14,352.9397,6.732,114,Two gangsters and the woman they love try to s...,"Comedy, Science Fiction, Crime"
3,1084187,Pretty Lethal,2026-03-13,311.0726,6.765,162,A troupe of ballerinas find themselves fightin...,"Action, Thriller, Music"
4,1290821,Shelter,2026-01-28,308.3847,6.751,445,A man living in self-imposed exile on a remote...,"Action, Crime, Thriller"


In [25]:
# Summary statistics
print("Summary Statistics:")

df_loaded.describe()

Summary Statistics:


,movie_id,popularity,vote_average,vote_count
count,4.000000e+01,40.000000,40.000000,40.00000
mean,1.101148e+06,170.410395,6.710725,1289.05000
std,3.419731e+05,188.784202,1.043590,3707.98682
min,8.353300e+04,65.985100,4.000000,1.00000
25%,8.725558e+05,77.024650,6.161250,59.00000
50%,1.169668e+06,96.918750,6.878500,333.50000
75%,1.312296e+06,180.426900,7.387500,854.75000
max,1.646787e+06,1165.126600,8.500000,21725.00000


## EDA 2 — Number of Movies per Genre

Since one movie can belong to multiple genres, I split the comma-separated genre strings and counted how many times each genre appears across all 40 movies.

From the output, **Thriller** is the most common genre with 15 movies, followed by **Comedy** and **Horror** with 11 each.

In [26]:
from collections import Counter

all_genres = []
for genres in df_loaded["genres"].dropna():
    for genre in genres.split(", "):
        all_genres.append(genre.strip())

genre_counts = pd.Series(Counter(all_genres)).sort_values(ascending=False)
print("Movies per genre:")
print(genre_counts)

Movies per genre:
Thriller           15
Comedy             11
Horror             11
Action             10
Science Fiction    10
Drama              10
Adventure           9
Crime               9
Mystery             7
Family              5
Animation           5
Romance             3
Fantasy             3
Music               1
dtype: int64


## EDA 3 — Checking for Missing Values

It's important to check if any columns have null/missing values before doing any further analysis or model building. I used `isnull().sum()` to count missing values per column.

In this dataset, all 8 columns have **0 missing values**, which means the data we collected from TMDB is clean and complete.

In [27]:
missing = df_loaded.isnull().sum()
print("Missing values per column:")
print(missing)

Missing values per column:
movie_id        0
title           0
release_date    0
popularity      0
vote_average    0
vote_count      0
overview        0
genres          0
dtype: int64


---

## Conclusion

In this notebook I successfully:
- ✅ Connected to the TMDB API and fetched **40 movies** across 2 pages
- ✅ Extracted 8 meaningful fields per movie including genres, ratings, and popularity
- ✅ Stored the data in a **SQLite database** table called `movies`
- ✅ Loaded it back and performed **EDA** using pandas

**Key Observations:**
- Thriller, Comedy, and Horror are the most popular genres in current trending movies
- The dataset has no missing values, so no cleaning was required
- Vote counts vary a lot — some movies have 21,000+ votes while others have very few, showing a big difference in popularity

This pipeline can easily be extended to fetch more pages, add filters like genre or year, or even schedule it to run daily to track trends over time.